# Monitoring Schedules
This notebook runs monitors against the baselines from `6_Monitoring`:
1. Sends held-out production-split images to both endpoints, tagged with inference IDs, and uploads their ground-truth labels.
2. Runs every monitor once on normal traffic, then again on degraded traffic (darkened and blurred), to check that problems are detected and normal traffic passes.
3. Leaves hourly schedules and CloudWatch alarms running.

| Monitor | Endpoint | Needs labels | Checks | Implementation |
|---|---|---|---|---|
| Model quality | classifier | yes | accuracy, precision, recall, F1, AUC | Model Monitor `ModelQualityMonitor` |
| Input and prediction drift | classifier | no | image size, brightness, contrast; predicted class and probability | Model Monitor `DefaultModelMonitor` with a record preprocessor |
| Input and prediction drift | keypoint | no | image size, brightness, contrast; predicted coordinates and eye distance | Model Monitor `DefaultModelMonitor` with a record preprocessor |
| Model quality (NME) | keypoint | yes | normalized mean error | Custom processing job in a SageMaker Pipeline (`src/monitoring/keypoint_quality.py`) |

In [1]:
# Basic imports
import io
import json
import time
import uuid
from datetime import datetime, timedelta, timezone

# Third party core
import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import roc_auc_score
from sqlalchemy import create_engine
from tqdm import tqdm

# Sagemaker imports
import sagemaker
from sagemaker.deserializers import CSVDeserializer, JSONDeserializer
from sagemaker.model_monitor import (
    CronExpressionGenerator,
    DefaultModelMonitor,
    EndpointInput,
    ModelQualityMonitor,
)
from sagemaker.predictor import Predictor
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.serializers import IdentitySerializer
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.functions import Join
from sagemaker.workflow.parameters import ParameterInteger, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.triggers import PipelineSchedule

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r

In [3]:
%store

Stored variables and their in-db values:
baseline_prefix                            -> 'cat-landmarks-project/monitoring/f0d83606-c3c6-4b
bucket                                     -> 'sagemaker-us-east-1-298748835671'
cls_endpoint_name                          -> 'cls-f0d83606-c3c6-4b88-a55b-63efac0bccfc'
cls_model_data_url                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
cls_model_package_arn                      -> 'arn:aws:sagemaker:us-east-1:298748835671:model-pa
data_capture_prefix                        -> 'cat-landmarks-project/monitoring/f0d83606-c3c6-4b
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                        -> True
kp_endpoint_name                           -> 'keypoint-regression-f0d83606-c3c6-4b88-a55b-63efa
kp_model_data_url                          -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
kp_model_package_arn                     

In [4]:
bucket = bucket
project_prefix = project_prefix

s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
cloudwatch = boto_session.client("cloudwatch", region_name=region)

# SQL connection specification
engine = create_engine(
    f"awsathena+rest://@athena.{region}.amazonaws.com:443/"
    f"{database_name}"
    f"?s3_staging_dir={s3_staging_dir}"
)

# Baselines from 6_Monitoring
cls_baseline_prefix = f"{baseline_prefix}/classifier"
kp_baseline_prefix = f"{baseline_prefix}/keypoint"

# Ground truth, monitoring constraints and reports
cls_ground_truth_prefix = f"{monitoring_prefix}/ground-truth/classifier"
kp_ground_truth_prefix = f"{monitoring_prefix}/ground-truth/keypoint"
constraints_prefix = f"{monitoring_prefix}/constraints"
reports_prefix = f"{monitoring_prefix}/reports"

# Data capture written by each endpoint
cls_capture_prefix = f"{data_capture_prefix}/classifier/{cls_endpoint_name}"
kp_capture_prefix = f"{data_capture_prefix}/keypoint_regressor/{kp_endpoint_name}"

MONITOR_INSTANCE_TYPE = "ml.m5.xlarge"
KP_METRIC_NAMESPACE = "CatKeypointPipeline/KeypointQuality"
run_id = uuid.uuid4().hex[:8]

KP_NAMES = [
    "left_eye", "right_eye", "mouth",
    "left_ear_1", "left_ear_2", "left_ear_3",
    "right_ear_1", "right_ear_2", "right_ear_3",
]
INPUT_STAT_NAMES = ["width", "height", "aspect_ratio", "brightness", "contrast"]

In [5]:
def read_s3_bytes(s3_uri: str) -> bytes:
    bucket_name, key = s3_uri.replace("s3://", "").split("/", 1)
    return s3.get_object(Bucket=bucket_name, Key=key)["Body"].read()


def read_s3_json(key: str) -> dict:
    return json.loads(read_s3_bytes(f"s3://{bucket}/{key}"))


def write_s3_json(key: str, body: dict) -> str:
    s3.put_object(Bucket=bucket, Key=key, Body=json.dumps(body, indent=2).encode())
    return f"s3://{bucket}/{key}"


def keypoint_nme(pred_norm, gt_norm, width, height) -> np.ndarray:
    """Per-keypoint pixel error divided by the ground-truth inter-ocular distance.
    Same definition as src/keypoint_regression/evaluation.py."""
    scale = np.array([width, height], dtype=float)
    pred_px = np.asarray(pred_norm, dtype=float).reshape(-1, 2) * scale
    gt_px = np.asarray(gt_norm, dtype=float).reshape(-1, 2) * scale
    inter_ocular = np.linalg.norm(gt_px[0] - gt_px[1])
    if inter_ocular == 0:
        return np.full(len(KP_NAMES), np.nan)
    return np.linalg.norm(pred_px - gt_px, axis=1) / inter_ocular

# Monitoring Thresholds
Model Monitor's suggested constraints use the baseline value itself as the threshold. Any monitoring window that scores slightly below the test set would then raise a violation, even when the drop is sampling noise.

Each threshold here is the baseline value moved by three bootstrap standard errors, estimated from the baseline traffic at the size of one monitoring window. A violation therefore means a change larger than that sample size can explain. Drift checks keep Model Monitor's default distribution-distance threshold.

In [6]:
N_CLS_REQUESTS = 500           # labelled classifier requests per normal-traffic window
N_KP_REQUESTS = 300            # labelled keypoint requests per normal-traffic window
STANDARD_ERRORS = 3            # slack before a metric counts as a violation
BOOTSTRAP_SAMPLES = 1000


def classifier_metrics(labels, preds, probs) -> dict:
    """Same metrics Model Monitor computes for binary classification, cat = positive class."""
    tp = np.sum((preds == 1) & (labels == 1)); tn = np.sum((preds == 0) & (labels == 0))
    fp = np.sum((preds == 1) & (labels == 0)); fn = np.sum((preds == 0) & (labels == 1))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f_beta = lambda b: (1 + b**2) * precision * recall / max(b**2 * precision + recall, 1e-12)
    return {
        "accuracy": (tp + tn) / len(labels),
        "precision": precision,
        "recall": recall,
        "true_positive_rate": recall,
        "true_negative_rate": tn / max(tn + fp, 1),
        "false_positive_rate": fp / max(tn + fp, 1),
        "false_negative_rate": fn / max(tp + fn, 1),
        "f0_5": f_beta(0.5),
        "f1": f_beta(1),
        "f2": f_beta(2),
        "auc": roc_auc_score(labels, probs),
    }


def keypoint_metrics(nme) -> dict:
    return {
        "nme_mean": nme.mean(),
        "nme_median": np.median(nme),
        "nme_p95": np.percentile(nme, 95),
        "failure_rate_at_0_1": (nme > 0.1).mean(),
    }


def bootstrap_standard_errors(metric_fn, arrays, n, seed=0) -> dict:
    """Standard error of each metric for a monitoring window of n records."""
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(BOOTSTRAP_SAMPLES):
        idx = rng.integers(0, len(arrays[0]), n)
        samples.append(metric_fn(*(a[idx] for a in arrays)))
    return pd.DataFrame(samples).std().to_dict()


# Per-request baseline results logged by 6_Monitoring
cls_log = pd.read_csv(io.BytesIO(read_s3_bytes(f"s3://{bucket}/{cls_baseline_prefix}/inference-log.csv")))
kp_log = pd.read_csv(io.BytesIO(read_s3_bytes(f"s3://{bucket}/{kp_baseline_prefix}/inference-log.csv")))

cls_standard_errors = bootstrap_standard_errors(
    classifier_metrics,
    [cls_log["label"].to_numpy(), cls_log["prediction"].to_numpy(), cls_log["probability"].to_numpy()],
    N_CLS_REQUESTS,
)
kp_standard_errors = bootstrap_standard_errors(keypoint_metrics, [kp_log["nme"].dropna().to_numpy()], N_KP_REQUESTS)

In [7]:
def relax(constraints: dict, standard_errors: dict) -> pd.DataFrame:
    """Move each threshold STANDARD_ERRORS standard errors in the lenient direction, in place."""
    rows = []
    for name, constraint in constraints.items():
        baseline = constraint["threshold"]
        if name in standard_errors:
            slack = STANDARD_ERRORS * standard_errors[name]
            constraint["threshold"] = baseline - slack if constraint["comparison_operator"] == "LessThanThreshold" else baseline + slack
        rows.append({"metric": name, "baseline": baseline, "threshold": constraint["threshold"],
                     "violation_if": constraint["comparison_operator"]})
    return pd.DataFrame(rows).set_index("metric")


cls_constraints = read_s3_json(f"{cls_baseline_prefix}/model-quality/constraints.json")
display(relax(cls_constraints["binary_classification_constraints"], cls_standard_errors).round(4))
cls_constraints_uri = write_s3_json(f"{constraints_prefix}/classifier/constraints.json", cls_constraints)

kp_constraints = read_s3_json(f"{kp_baseline_prefix}/model-quality/constraints.json")
display(relax(kp_constraints["keypoint_constraints"], kp_standard_errors).round(4))
kp_constraints_uri = write_s3_json(f"{constraints_prefix}/keypoint/constraints.json", kp_constraints)

# Drift checks use the baseline's own constraints (distribution-distance threshold below)
cls_drift_constraints = read_s3_json(f"{cls_baseline_prefix}/prediction-drift/constraints.json")
drift_threshold = cls_drift_constraints["monitoring_config"]["distribution_constraints"]["comparison_threshold"]
print("Drift threshold (distribution distance):", drift_threshold)

,baseline,threshold,violation_if
metric,,,
recall,0.9649,0.9330,LessThanThreshold
precision,0.9682,0.9384,LessThanThreshold
accuracy,0.9619,0.9373,LessThanThreshold
true_positive_rate,0.9649,0.9330,LessThanThreshold
true_negative_rate,0.9579,0.9193,LessThanThreshold
false_positive_rate,0.0421,0.0807,GreaterThanThreshold
false_negative_rate,0.0351,0.0670,GreaterThanThreshold
auc,0.9920,0.9827,LessThanThreshold
f0_5,0.9675,0.9427,LessThanThreshold


,baseline,threshold,violation_if
metric,,,
nme_mean,0.4026,0.4727,GreaterThanThreshold
nme_median,0.2868,0.3211,GreaterThanThreshold
nme_p95,1.0582,1.4841,GreaterThanThreshold
failure_rate_at_0_1,0.9895,1.0068,GreaterThanThreshold


Drift threshold (distribution distance): 0.1


# Monitors
Model Monitor allows one schedule of each type per endpoint, so the same monitor objects are reused for the one-time runs below and then for the hourly schedules.

In [8]:
def make_quality_monitor():
    return ModelQualityMonitor(
        role=role,
        instance_count=1,
        instance_type=MONITOR_INSTANCE_TYPE,
        volume_size_in_gb=20,
        max_runtime_in_seconds=1800,
        sagemaker_session=sagemaker_session,
    )


def make_data_monitor():
    return DefaultModelMonitor(
        role=role,
        instance_count=1,
        instance_type=MONITOR_INSTANCE_TYPE,
        volume_size_in_gb=20,
        max_runtime_in_seconds=1800,
        sagemaker_session=sagemaker_session,
    )


cls_quality_monitor = make_quality_monitor()
cls_drift_monitor = make_data_monitor()
kp_drift_monitor = make_data_monitor()


def schedule_cls_quality(name, cron, start=None, end=None):
    """Classifier model quality: captured predictions joined with uploaded ground truth."""
    cls_quality_monitor.create_monitoring_schedule(
        monitor_schedule_name=name,
        endpoint_input=EndpointInput(
            endpoint_name=cls_endpoint_name,
            destination="/opt/ml/processing/input_data",
            # CSV response columns: 0 = predicted class, 1 = cat probability
            inference_attribute="0",
            probability_attribute="1",
            probability_threshold_attribute=0.5,
        ),
        ground_truth_input=f"s3://{bucket}/{cls_ground_truth_prefix}",
        problem_type="BinaryClassification",
        constraints=cls_constraints_uri,
        output_s3_uri=f"s3://{bucket}/{reports_prefix}/classifier-model-quality",
        schedule_cron_expression=cron,
        enable_cloudwatch_metrics=True,
        data_analysis_start_time=start,
        data_analysis_end_time=end,
    )


def schedule_drift(monitor, endpoint_name, preprocessor, baseline_dir, report_name, name, cron, start=None, end=None):
    """Input and prediction drift: record features compared with the baseline distributions."""
    monitor.create_monitoring_schedule(
        monitor_schedule_name=name,
        endpoint_input=EndpointInput(
            endpoint_name=endpoint_name,
            destination="/opt/ml/processing/input/endpoint",
        ),
        record_preprocessor_script=preprocessor,
        statistics=f"s3://{bucket}/{baseline_dir}/prediction-drift/statistics.json",
        constraints=f"s3://{bucket}/{baseline_dir}/prediction-drift/constraints.json",
        output_s3_uri=f"s3://{bucket}/{reports_prefix}/{report_name}",
        schedule_cron_expression=cron,
        enable_cloudwatch_metrics=True,
        data_analysis_start_time=start,
        data_analysis_end_time=end,
    )


def schedule_cls_drift(name, cron, start=None, end=None):
    schedule_drift(cls_drift_monitor, cls_endpoint_name, "src/monitoring/classifier_record_preprocessor.py",
                   cls_baseline_prefix, "classifier-drift", name, cron, start, end)


def schedule_kp_drift(name, cron, start=None, end=None):
    schedule_drift(kp_drift_monitor, kp_endpoint_name, "src/monitoring/keypoint_record_preprocessor.py",
                   kp_baseline_prefix, "keypoint-drift", name, cron, start, end)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


## Keypoint quality pipeline
`src/monitoring/keypoint_quality.py` runs as a one-step SageMaker Pipeline. With no parameters it checks the last hour, which is what the hourly schedule uses. `StartTime` and `EndTime` select an exact window.

In [9]:
pipeline_session = PipelineSession()

processing_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.1.0",
    py_version="py310",
    instance_type=MONITOR_INSTANCE_TYPE,
    image_scope="training",
)

start_time_param = ParameterString(name="StartTime", default_value="auto")
end_time_param = ParameterString(name="EndTime", default_value="auto")
lookback_hours_param = ParameterInteger(name="LookbackHours", default_value=1)

kp_quality_args = ScriptProcessor(
    command=["python3"],
    image_uri=processing_image_uri,
    role=role,
    instance_type=MONITOR_INSTANCE_TYPE,
    instance_count=1,
    base_job_name="kp-quality-monitor",
    sagemaker_session=pipeline_session,
).run(
    code="src/monitoring/keypoint_quality.py",
    inputs=[
        ProcessingInput(source=f"s3://{bucket}/{kp_capture_prefix}", destination="/opt/ml/processing/input/capture"),
        ProcessingInput(source=f"s3://{bucket}/{kp_ground_truth_prefix}", destination="/opt/ml/processing/input/ground_truth"),
        ProcessingInput(source=kp_constraints_uri, destination="/opt/ml/processing/input/constraints"),
    ],
    outputs=[
        ProcessingOutput(
            output_name="report",
            source="/opt/ml/processing/report",
            # One report folder per pipeline execution
            destination=Join(on="/", values=["s3:/", bucket, reports_prefix, "keypoint-quality",
                                             ExecutionVariables.PIPELINE_EXECUTION_ID]),
        ),
    ],
    arguments=[
        "--start-time", start_time_param,
        "--end-time", end_time_param,
        "--lookback-hours", lookback_hours_param.to_string(),
        "--endpoint-name", kp_endpoint_name,
        "--region", region,
        "--metric-namespace", KP_METRIC_NAMESPACE,
    ],
)

kp_quality_pipeline = Pipeline(
    name="KeypointQualityMonitor",
    parameters=[start_time_param, end_time_param, lookback_hours_param],
    steps=[ProcessingStep(name="KeypointQualityCheck", step_args=kp_quality_args)],
    sagemaker_session=pipeline_session,
)
kp_quality_pipeline.upsert(role_arn=role)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


{'PipelineArn': 'arn:aws:sagemaker:us-east-1:298748835671:pipeline/KeypointQualityMonitor',
 'ResponseMetadata': {'RequestId': '5687222f-43d8-481c-9424-13316f16983c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '5687222f-43d8-481c-9424-13316f16983c',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '90',
   'date': 'Sun, 11 Oct 2026 19:48:20 GMT'},
  'RetryAttempts': 0}}

# Production Traffic and Ground Truth
The production split has been held out since ingestion: no model trained or was evaluated on it. Each request carries an inference ID. Labels are uploaded to S3 in Model Monitor's ground-truth format, in an hourly folder for the time of upload. Model Monitor uses the inference IDs to join each label with its captured prediction.

In [10]:
cls_predictor = Predictor(
    endpoint_name=cls_endpoint_name,
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=CSVDeserializer(),
)

kp_predictor = Predictor(
    endpoint_name=kp_endpoint_name,
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=JSONDeserializer(),
)

# Classifier: production split of the training manifest, from every image source
training_manifest = pd.read_parquet(io.BytesIO(read_s3_bytes(f"s3://{bucket}/{training_manifest_remote_path}")))
cls_production = training_manifest[training_manifest["split"] == "production"]

# Keypoint: production split of the landmark annotations, original images only
kp_columns = ", ".join(f"{kp}_{axis}" for kp in KP_NAMES for axis in ("x", "y"))
kp_production = pd.read_sql(f"""
    SELECT remote_path, width, height, {kp_columns}
    FROM {database_name}.{landmarks_table}
    WHERE split = 'production' AND version = 'raw'
""", engine)

# Disjoint samples: normal traffic, then twice as much degraded traffic
cls_sample = cls_production.sample(n=3 * N_CLS_REQUESTS, random_state=42)
kp_sample = kp_production.sample(n=3 * N_KP_REQUESTS, random_state=42)
cls_normal, cls_degraded = cls_sample.iloc[:N_CLS_REQUESTS], cls_sample.iloc[N_CLS_REQUESTS:]
kp_normal, kp_degraded = kp_sample.iloc[:N_KP_REQUESTS], kp_sample.iloc[N_KP_REQUESTS:]

print(f"Classifier production pool: {len(cls_production)} | keypoint production pool: {len(kp_production)}")
cls_sample.groupby(["source", "label"]).size().unstack("label")

Classifier production pool: 18588 | keypoint production pool: 7999


label,0,1
source,,
coco,486.0,143.0
kaggle_cats,NaN,646.0
oxford_pets,158.0,67.0


In [11]:
def degrade(image_bytes: bytes) -> bytes:
    """Simulated camera problem: underexposed and out of focus. Geometry is unchanged,
    so the keypoint ground truth still applies."""
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    image = ImageEnhance.Brightness(image).enhance(0.35)
    image = image.filter(ImageFilter.GaussianBlur(radius=max(image.size) / 60))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=90)
    return buffer.getvalue()


def invoke(predictor, s3_uri: str, degraded: bool):
    payload = read_s3_bytes(s3_uri)
    if degraded:
        payload = degrade(payload)
    inference_id = str(uuid.uuid4())
    return inference_id, predictor.predict(payload, inference_id=inference_id)


def upload_ground_truth(prefix: str, records) -> str:
    """records: (inference_id, data, encoding). Written in Model Monitor's ground-truth format."""
    now = datetime.now(timezone.utc)
    lines = [
        json.dumps({
            "groundTruthData": {"data": data, "encoding": encoding},
            "eventMetadata": {"eventId": inference_id},
            "eventVersion": "0",
        })
        for inference_id, data, encoding in records
    ]
    key = f"{prefix}/{now:%Y/%m/%d/%H}/ground-truth-{uuid.uuid4().hex}.jsonl"
    s3.put_object(Bucket=bucket, Key=key, Body="\n".join(lines).encode())
    return key


def send_traffic(cls_rows: pd.DataFrame, kp_rows: pd.DataFrame, degraded: bool) -> dict:
    """Send both batches, upload their ground truth, and score the responses directly
    so the monitors' results can be checked against a known answer."""
    start = datetime.now(timezone.utc)

    cls_results = []
    for uri, label in tqdm(zip(cls_rows["remote_path"], cls_rows["label"]), total=len(cls_rows),
                           desc="Classifier traffic", unit="image"):
        inference_id, response = invoke(cls_predictor, uri, degraded)
        values = response[0]
        cls_results.append({
            "inference_id": inference_id,
            "label": int(label),
            "prediction": int(values[0]),
            "probability": float(values[1]),
            **{name: float(value) for name, value in zip(INPUT_STAT_NAMES, values[2:])},
        })

    kp_results = []
    for _, row in tqdm(kp_rows.iterrows(), total=len(kp_rows), desc="Keypoint traffic", unit="image"):
        inference_id, response = invoke(kp_predictor, row["remote_path"], degraded)
        gt_norm = [row[f"{kp}_x"] / row["width"] if axis == "x" else row[f"{kp}_y"] / row["height"]
                   for kp in KP_NAMES for axis in ("x", "y")]
        pred_norm = [response[kp][axis] for kp in KP_NAMES for axis in ("x", "y")]
        kp_results.append({
            "inference_id": inference_id,
            "ground_truth": {"keypoints_norm": gt_norm, "width": int(row["width"]), "height": int(row["height"])},
            "nme": float(np.mean(keypoint_nme(pred_norm, gt_norm, row["width"], row["height"]))),
            **response["input_stats"],
        })

    upload_ground_truth(cls_ground_truth_prefix,
                        [(r["inference_id"], str(r["label"]), "CSV") for r in cls_results])
    upload_ground_truth(kp_ground_truth_prefix,
                        [(r["inference_id"], json.dumps(r["ground_truth"]), "JSON") for r in kp_results])

    return {
        "start": start,
        "end": datetime.now(timezone.utc),
        "cls": pd.DataFrame(cls_results),
        "kp": pd.DataFrame(kp_results).drop(columns="ground_truth"),
    }


def wait_for_capture(capture_prefix: str, inference_ids, timeout_minutes: int = 20):
    """Data capture lands in S3 a few minutes after the requests. Wait until every request is there."""
    pending, seen_keys = set(inference_ids), set()
    deadline = time.time() + timeout_minutes * 60
    while pending and time.time() < deadline:
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=capture_prefix):
            for obj in page.get("Contents", []):
                if obj["Key"] in seen_keys:
                    continue
                seen_keys.add(obj["Key"])
                for line in read_s3_bytes(f"s3://{bucket}/{obj['Key']}").decode().splitlines():
                    if line.strip():
                        pending.discard(json.loads(line)["eventMetadata"].get("inferenceId"))
        if pending:
            time.sleep(30)
    if pending:
        raise TimeoutError(f"{len(pending)} requests not captured under {capture_prefix} after {timeout_minutes} minutes")


def direct_scores(traffic: dict) -> dict:
    """What the monitors should see: metrics computed straight from the responses."""
    cls_df, kp_df = traffic["cls"], traffic["kp"]
    cls_scores = classifier_metrics(cls_df["label"].to_numpy(), cls_df["prediction"].to_numpy(), cls_df["probability"].to_numpy())
    return {
        "classifier accuracy": cls_scores["accuracy"],
        "classifier f1": cls_scores["f1"],
        "classifier mean brightness": cls_df["brightness"].mean(),
        "keypoint nme_mean": kp_df["nme"].mean(),
        "keypoint mean brightness": kp_df["brightness"].mean(),
    }

In [12]:
TERMINAL_STATUSES = {"Completed", "CompletedWithViolations", "Failed", "Stopped"}


def wait_for_monitoring_execution(schedule_name: str, timeout_minutes: int = 60) -> dict:
    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        summaries = sagemaker_client.list_monitoring_executions(
            MonitoringScheduleName=schedule_name)["MonitoringExecutionSummaries"]
        if summaries and summaries[0]["MonitoringExecutionStatus"] in TERMINAL_STATUSES:
            return summaries[0]
        time.sleep(60)
    raise TimeoutError(f"No finished execution for {schedule_name} after {timeout_minutes} minutes")


def violation_names(violations) -> str:
    return ", ".join(sorted({v.get("feature_name") or v.get("metric_name") for v in violations})) or "-"


def run_monitors_once(scenario: str, traffic: dict) -> pd.DataFrame:
    """Run every monitor once on this scenario's traffic and collect the outcomes.

    Model Monitor's one-time runs analyze the last hour, so a window can also include
    earlier traffic. The keypoint NME check uses the exact window of this scenario."""
    schedule_cls_quality(f"cls-quality-{scenario}-{run_id}", CronExpressionGenerator.now(), "-PT1H", "-PT0H")
    schedule_cls_drift(f"cls-drift-{scenario}-{run_id}", CronExpressionGenerator.now(), "-PT1H", "-PT0H")
    schedule_kp_drift(f"kp-drift-{scenario}-{run_id}", CronExpressionGenerator.now(), "-PT1H", "-PT0H")

    margin = timedelta(minutes=1)
    kp_execution = kp_quality_pipeline.start(parameters={
        "StartTime": (traffic["start"] - margin).isoformat(),
        "EndTime": (traffic["end"] + margin).isoformat(),
    })

    rows = []
    for name, monitor in [("classifier model quality", cls_quality_monitor),
                          ("classifier input/prediction drift", cls_drift_monitor),
                          ("keypoint input/prediction drift", kp_drift_monitor)]:
        summary = wait_for_monitoring_execution(monitor.monitoring_schedule_name)
        status = summary["MonitoringExecutionStatus"]
        violations = []
        if status in ("Completed", "CompletedWithViolations"):
            report = monitor.list_executions()[-1].constraint_violations()
            violations = [] if report is None else report.body_dict["violations"]
        rows.append({"scenario": scenario, "monitor": name, "status": status,
                     "violations": violation_names(violations),
                     "failure_reason": summary.get("FailureReason", "")})
        # Frees the endpoint for the next schedule of this type
        monitor.delete_monitoring_schedule()

    kp_execution.wait(delay=30, max_attempts=60)
    report_key = f"{reports_prefix}/keypoint-quality/{kp_execution.arn.split('/')[-1]}"
    kp_violations = read_s3_json(f"{report_key}/constraint_violations.json")["violations"]
    kp_statistics = read_s3_json(f"{report_key}/statistics.json")
    rows.append({"scenario": scenario, "monitor": "keypoint model quality (NME)",
                 "status": "CompletedWithViolations" if kp_violations else "Completed",
                 "violations": violation_names(kp_violations),
                 "failure_reason": f"{kp_statistics['dataset']['item_count']} records scored"})
    return pd.DataFrame(rows)

## Normal traffic
Held-out production images, unmodified. Expected result: no model-quality violations and no drift.

In [13]:
normal_traffic = send_traffic(cls_normal, kp_normal, degraded=False)
wait_for_capture(cls_capture_prefix, normal_traffic["cls"]["inference_id"])
wait_for_capture(kp_capture_prefix, normal_traffic["kp"]["inference_id"])

normal_results = run_monitors_once("normal", normal_traffic)
normal_results

Keypoint traffic: 100%|██████████| 300/300 [00:26<00:00, 11.40image/s]
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-quality-normal-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-drift-normal-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: kp-drift-normal-cc21b50c
INFO:sagemaker:Deleting Monitoring Schedule with name: cls-quality-normal-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Deleting Model Quality Job Definition with name: model-quality-job-definition-2026-10-11-19-50-52-111
INFO:sagemaker:Deleting Monitoring Schedule with name: cls-drift-normal-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Deleting Data Quality Job Definition with name: data-quality-job-definition-2026-10-11-19-50-52-759
INFO:sagemaker:Deleting Monitoring Schedule with name: kp-drift-normal-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Deleting

,scenario,monitor,status,violations,failure_reason
0,normal,classifier model quality,Failed,-,"No S3 objects found under S3 URL ""s3://sagemak..."
1,normal,classifier input/prediction drift,Failed,-,Job inputs had no data
2,normal,keypoint input/prediction drift,Failed,-,Job inputs had no data
3,normal,keypoint model quality (NME),Completed,-,300 records scored


## Degraded traffic
The same kind of images, darkened to 35% brightness and blurred: a simulated camera or lighting failure. The batch is twice the size of the normal one. Model Monitor's one-hour window also contains the normal batch, so about two thirds of the records it analyzes are degraded. The keypoint NME check scores the degraded batch alone.

In [14]:
degraded_traffic = send_traffic(cls_degraded, kp_degraded, degraded=True)
wait_for_capture(cls_capture_prefix, degraded_traffic["cls"]["inference_id"])
wait_for_capture(kp_capture_prefix, degraded_traffic["kp"]["inference_id"])

degraded_results = run_monitors_once("degraded", degraded_traffic)
degraded_results

Keypoint traffic: 100%|██████████| 600/600 [01:04<00:00,  9.29image/s]
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-quality-degraded-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-drift-degraded-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: kp-drift-degraded-cc21b50c
INFO:sagemaker:Deleting Monitoring Schedule with name: cls-quality-degraded-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Deleting Model Quality Job Definition with name: model-quality-job-definition-2026-10-11-20-02-21-016
INFO:sagemaker:Deleting Monitoring Schedule with name: cls-drift-degraded-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Deleting Data Quality Job Definition with name: data-quality-job-definition-2026-10-11-20-02-21-788
INFO:sagemaker:Deleting Monitoring Schedule with name: kp-drift-degraded-cc21b50c
INFO:sagemaker.model_monitor.model_monitor

,scenario,monitor,status,violations,failure_reason
0,degraded,classifier model quality,Completed,-,
1,degraded,classifier input/prediction drift,CompletedWithViolations,"brightness, contrast",
2,degraded,keypoint input/prediction drift,CompletedWithViolations,"brightness, contrast",
3,degraded,keypoint model quality (NME),CompletedWithViolations,"nme_mean, nme_median, nme_p95",600 records scored


## Results
What each monitor reported, next to metrics computed directly from the endpoint responses. The direct metrics are the answer the monitors should reach.

In [15]:
display(pd.concat([normal_results, degraded_results]).set_index(["monitor", "scenario"]).sort_index())

pd.DataFrame({
    "normal": direct_scores(normal_traffic),
    "degraded": direct_scores(degraded_traffic),
    "threshold": {
        "classifier accuracy": cls_constraints["binary_classification_constraints"]["accuracy"]["threshold"],
        "classifier f1": cls_constraints["binary_classification_constraints"]["f1"]["threshold"],
        "keypoint nme_mean": kp_constraints["keypoint_constraints"]["nme_mean"]["threshold"],
    },
}).round(4)

status  \
monitor                           scenario                            
classifier input/prediction drift degraded  CompletedWithViolations   
                                  normal                     Failed   
classifier model quality          degraded                Completed   
                                  normal                     Failed   
keypoint input/prediction drift   degraded  CompletedWithViolations   
                                  normal                     Failed   
keypoint model quality (NME)      degraded  CompletedWithViolations   
                                  normal                  Completed   

                                                               violations  \
monitor                           scenario                                  
classifier input/prediction drift degraded           brightness, contrast   
                                  normal                                -   
classifier model quality          degraded                              -   
                                  normal                                -   
keypoint input/prediction drift   degraded           brightness, contrast   
                                  normal                                -   
keypoint model quality (NME)      degraded  nme_mean, nme_median, nme_p95   
                                  normal                                -   

                                                                               failure_reason  
monitor                           scenario                                                     
classifier input/prediction drift degraded                                                     
                                  normal                               Job inputs had no data  
classifier model quality          degraded                                                     
                                  normal    No S3 objects found under S3 URL "s3://sagemak...  
keypoint input/prediction drift   degraded                                                     
                                  normal                               Job inputs had no data  
keypoint model quality (NME)      degraded                                 600 records scored  
                                  normal                                   300 records scored

,normal,degraded,threshold
classifier accuracy,0.9500,0.5530,0.9373
classifier f1,0.9567,0.3800,0.9444
classifier mean brightness,0.4393,0.1520,NaN
keypoint nme_mean,0.4223,2.0334,0.4727
keypoint mean brightness,0.4303,0.1498,NaN


# Hourly Monitoring
The same monitors as hourly schedules, and CloudWatch alarms on the metrics they publish.

In [16]:
endpoint_suffix = run_id

cls_quality_schedule = f"cls-quality-hourly-{endpoint_suffix}"
cls_drift_schedule = f"cls-drift-hourly-{endpoint_suffix}"
kp_drift_schedule = f"kp-drift-hourly-{endpoint_suffix}"

schedule_cls_quality(cls_quality_schedule, CronExpressionGenerator.hourly())
schedule_cls_drift(cls_drift_schedule, CronExpressionGenerator.hourly())
schedule_kp_drift(kp_drift_schedule, CronExpressionGenerator.hourly())

for name in (cls_quality_schedule, cls_drift_schedule, kp_drift_schedule):
    print(name, sagemaker_client.describe_monitoring_schedule(MonitoringScheduleName=name)["MonitoringScheduleStatus"])

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-quality-hourly-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-drift-hourly-cc21b50c
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: kp-drift-hourly-cc21b50c


cls-quality-hourly-cc21b50c Pending
cls-drift-hourly-cc21b50c Pending
kp-drift-hourly-cc21b50c Pending


In [17]:
# EventBridge Scheduler starts the keypoint quality pipeline every hour. It needs an IAM role
# it can assume (trust: scheduler.amazonaws.com; permission: sagemaker:StartPipelineExecution).
# The README's "Scheduling the keypoint quality check" section creates it. Set its ARN here.
SCHEDULER_ROLE_ARN = None

if SCHEDULER_ROLE_ARN:
    kp_quality_trigger_arns = kp_quality_pipeline.put_triggers(
        triggers=[PipelineSchedule(name="keypoint-quality-hourly", rate=(1, "hour"))],
        role_arn=SCHEDULER_ROLE_ARN,
    )
    print("Hourly schedule:", kp_quality_trigger_arns)
else:
    print("SCHEDULER_ROLE_ARN is not set, so the keypoint quality check is not scheduled. "
          "Its one-time runs above are unaffected.")

SCHEDULER_ROLE_ARN is not set, so the keypoint quality check is not scheduled. Its one-time runs above are unaffected.


## Alarms
Alarms notify an SNS topic. To receive email, set `ALERT_EMAIL` and confirm the subscription email AWS sends.

In [18]:
ALERT_EMAIL = "safriedm39@gmail.com"

sns = boto_session.client("sns", region_name=region)
try:
    alarm_topic_arn = sns.create_topic(Name="cat-pipeline-model-monitoring-alerts")["TopicArn"]
    if ALERT_EMAIL:
        sns.subscribe(TopicArn=alarm_topic_arn, Protocol="email", Endpoint=ALERT_EMAIL)
    alarm_actions = [alarm_topic_arn]
except ClientError as e:
    if e.response["Error"]["Code"] not in ("AccessDenied", "AuthorizationError"):
        raise
    # The alarms still change state in CloudWatch; they just notify no one
    print(f"Cannot create the SNS topic ({e.response['Error']['Code']}). "
          "Grant the execution role sns:CreateTopic and sns:Subscribe to enable notifications.")
    alarm_actions = []


def put_alarm(name, namespace, metric, dimensions, threshold, comparison, description):
    cloudwatch.put_metric_alarm(
        AlarmName=name,
        AlarmDescription=description,
        Namespace=namespace,
        MetricName=metric,
        Dimensions=[{"Name": key, "Value": value} for key, value in dimensions.items()],
        Statistic="Average",
        Period=3600,
        EvaluationPeriods=1,
        Threshold=threshold,
        ComparisonOperator=comparison,
        TreatMissingData="notBreaching",
        AlarmActions=alarm_actions,
    )
    return name


MODEL_METRICS = "aws/sagemaker/Endpoints/model-metrics"
DATA_METRICS = "aws/sagemaker/Endpoints/data-metrics"

alarm_names = [
    put_alarm(f"cls-f1-{endpoint_suffix}", MODEL_METRICS, "f1",
              {"Endpoint": cls_endpoint_name, "MonitoringSchedule": cls_quality_schedule},
              cls_constraints["binary_classification_constraints"]["f1"]["threshold"], "LessThanThreshold",
              "Classifier F1 below the monitoring threshold"),
    put_alarm(f"cls-input-brightness-drift-{endpoint_suffix}", DATA_METRICS, "feature_baseline_drift_brightness",
              {"Endpoint": cls_endpoint_name, "MonitoringSchedule": cls_drift_schedule},
              drift_threshold, "GreaterThanThreshold", "Classifier input brightness drifted from the baseline"),
    put_alarm(f"cls-probability-drift-{endpoint_suffix}", DATA_METRICS, "feature_baseline_drift_probability",
              {"Endpoint": cls_endpoint_name, "MonitoringSchedule": cls_drift_schedule},
              drift_threshold, "GreaterThanThreshold", "Classifier cat-probability distribution drifted from the baseline"),
    put_alarm(f"kp-input-brightness-drift-{endpoint_suffix}", DATA_METRICS, "feature_baseline_drift_brightness",
              {"Endpoint": kp_endpoint_name, "MonitoringSchedule": kp_drift_schedule},
              drift_threshold, "GreaterThanThreshold", "Keypoint input brightness drifted from the baseline"),
    put_alarm(f"kp-eye-distance-drift-{endpoint_suffix}", DATA_METRICS, "feature_baseline_drift_pred_inter_ocular",
              {"Endpoint": kp_endpoint_name, "MonitoringSchedule": kp_drift_schedule},
              drift_threshold, "GreaterThanThreshold", "Predicted eye-distance distribution drifted from the baseline"),
    put_alarm(f"kp-nme-{endpoint_suffix}", KP_METRIC_NAMESPACE, "nme_mean",
              {"Endpoint": kp_endpoint_name},
              kp_constraints["keypoint_constraints"]["nme_mean"]["threshold"], "GreaterThanThreshold",
              "Keypoint NME above the monitoring threshold"),
]
alarm_names

Cannot create the SNS topic (AuthorizationError). Grant the execution role sns:CreateTopic and sns:Subscribe to enable notifications.


['cls-f1-cc21b50c',
 'cls-input-brightness-drift-cc21b50c',
 'cls-probability-drift-cc21b50c',
 'kp-input-brightness-drift-cc21b50c',
 'kp-eye-distance-drift-cc21b50c',
 'kp-nme-cc21b50c']

In [19]:
# The one-time runs above published these metrics. An alarm whose metric is missing here
# watches nothing, so this confirms each alarm's metric name. New metrics can take up to
# about 15 minutes to appear.
def published_metric_names(namespace, endpoint_name):
    names = set()
    for page in cloudwatch.get_paginator("list_metrics").paginate(
            Namespace=namespace, Dimensions=[{"Name": "Endpoint", "Value": endpoint_name}]):
        names.update(metric["MetricName"] for metric in page["Metrics"])
    return names


expected = [
    (MODEL_METRICS, cls_endpoint_name, "f1"),
    (DATA_METRICS, cls_endpoint_name, "feature_baseline_drift_brightness"),
    (DATA_METRICS, cls_endpoint_name, "feature_baseline_drift_probability"),
    (DATA_METRICS, kp_endpoint_name, "feature_baseline_drift_brightness"),
    (DATA_METRICS, kp_endpoint_name, "feature_baseline_drift_pred_inter_ocular"),
    (KP_METRIC_NAMESPACE, kp_endpoint_name, "nme_mean"),
]
pd.DataFrame([
    {"namespace": namespace, "endpoint": endpoint, "metric": metric,
     "published": metric in published_metric_names(namespace, endpoint)}
    for namespace, endpoint, metric in expected
])

,namespace,endpoint,metric,published
0,aws/sagemaker/Endpoints/model-metrics,cls-f0d83606-c3c6-4b88-a55b-63efac0bccfc,f1,True
1,aws/sagemaker/Endpoints/data-metrics,cls-f0d83606-c3c6-4b88-a55b-63efac0bccfc,feature_baseline_drift_brightness,True
2,aws/sagemaker/Endpoints/data-metrics,cls-f0d83606-c3c6-4b88-a55b-63efac0bccfc,feature_baseline_drift_probability,True
3,aws/sagemaker/Endpoints/data-metrics,keypoint-regression-f0d83606-c3c6-4b88-a55b-63...,feature_baseline_drift_brightness,True
4,aws/sagemaker/Endpoints/data-metrics,keypoint-regression-f0d83606-c3c6-4b88-a55b-63...,feature_baseline_drift_pred_inter_ocular,True
5,CatKeypointPipeline/KeypointQuality,keypoint-regression-f0d83606-c3c6-4b88-a55b-63...,nme_mean,True


In [20]:
%store cls_quality_schedule cls_drift_schedule kp_drift_schedule alarm_names

Stored 'cls_quality_schedule' (str)
Stored 'cls_drift_schedule' (str)
Stored 'kp_drift_schedule' (str)
Stored 'alarm_names' (list)


# Deleting monitors

In [21]:
""" for alarm in alarm_names:
    cloudwatch.delete_alarms(AlarmNames=[alarm])

cls_quality_monitor.delete_monitoring_schedule()
cls_drift_monitor.delete_monitoring_schedule()
kp_drift_monitor.delete_monitoring_schedule()

if SCHEDULER_ROLE_ARN:
    kp_quality_pipeline.delete_triggers(trigger_names=["keypoint-quality-hourly"])
kp_quality_pipeline.delete() """

' for alarm in alarm_names:\n    cloudwatch.delete_alarms(AlarmNames=[alarm])\n\ncls_quality_monitor.delete_monitoring_schedule()\ncls_drift_monitor.delete_monitoring_schedule()\nkp_drift_monitor.delete_monitoring_schedule()\n\nif SCHEDULER_ROLE_ARN:\n    kp_quality_pipeline.delete_triggers(trigger_names=["keypoint-quality-hourly"])\nkp_quality_pipeline.delete() '